In [ ]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

model_bundle = joblib.load(
    project_root / "models" / "jobshield_model.joblib"
)

model = model_bundle["model"]

vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]

feature_names = np.array(
    vectorizer.get_feature_names_out()
)

weights = classifier.coef_[0]

top_fraud_indices = np.argsort(weights)[-20:][::-1]
top_legitimate_indices = np.argsort(weights)[:20]

top_fraud_terms = pd.DataFrame({
    "term": feature_names[top_fraud_indices],
    "weight": weights[top_fraud_indices]
})

top_legitimate_terms = pd.DataFrame({
    "term": feature_names[top_legitimate_indices],
    "weight": weights[top_legitimate_indices]
})

print("Model loaded successfully")


In [ ]:
print(top_fraud_terms)

In [ ]:
print(top_legitimate_terms)

In [ ]:
from src.config import TARGET_COLUMN
from src.text_processing import combine_text_columns


train_df = pd.read_csv(
    project_root / "data" / "processed" / "train.csv"
)

train_text = combine_text_columns(train_df)
y_train = train_df[TARGET_COLUMN]

print("Training documents loaded:", len(train_text))

In [ ]:
import re


def inspect_term(term):
    pattern = rf"\b{re.escape(term)}\b"

    contains_term = train_text.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

    matching_targets = y_train.loc[
        contains_term
    ]

    document_count = int(contains_term.sum())
    fraudulent_count = int(matching_targets.sum())

    fraud_rate = (
        fraudulent_count / document_count
        if document_count > 0
        else 0
    )

    return {
        "term": term,
        "document_count": document_count,
        "fraudulent_count": fraudulent_count,
        "fraud_rate": fraud_rate
    }


terms_to_inspect = [
    "link",
    "earn",
    "cash",
    "vam",
    "engineering",
    "companies",
    "team",
    "valor"
]

term_analysis = pd.DataFrame([
    inspect_term(term)
    for term in terms_to_inspect
])

term_analysis["fraud_rate"] = (
    term_analysis["fraud_rate"] * 100
).round(2)

term_analysis

In [ ]:
sample_posting = pd.DataFrame([{
    "title": "Work From Home Data Entry",
    "company_profile": "",
    "description": (
        "Earn money immediately. Pay a registration "
        "fee to receive your appointment letter."
    ),
    "requirements": "No experience required",
    "benefits": "Guaranteed weekly income",
}])

combined_sample = model.named_steps[
    "combine_text"
].transform(sample_posting)

sample_vector = vectorizer.transform(
    combined_sample
)

contributions = (
    sample_vector
    .multiply(weights)
    .toarray()[0]
)

tfidf_values = sample_vector.toarray()[0]

used_indices = np.flatnonzero(tfidf_values)

local_explanation = pd.DataFrame({
    "term": feature_names[used_indices],
    "tfidf_value": tfidf_values[used_indices],
    "model_weight": weights[used_indices],
    "contribution": contributions[used_indices],
})

local_explanation = local_explanation.sort_values(
    "contribution",
    ascending=False
)

print(
    "Fraud score:",
    round(
        model.predict_proba(sample_posting)[0, 1],
        4
    )
)

local_explanation

In [ ]:
total_term_contribution = contributions.sum()
intercept = float(classifier.intercept_[0])

logit = intercept + total_term_contribution
manual_fraud_score = 1 / (1 + np.exp(-logit))

print(
    "Term contributions:",
    round(total_term_contribution, 4)
)
print(
    "Intercept:",
    round(intercept, 4)
)
print(
    "Final logit:",
    round(logit, 4)
)
print(
    "Manual fraud score:",
    round(manual_fraud_score, 4)
)
print(
    "Model fraud score:",
    round(model.predict_proba(sample_posting)[0, 1], 4)
)

In [ ]:
validation_df = pd.read_csv(
    project_root / "data" / "processed" / "validation.csv"
)

validation_text = combine_text_columns(
    validation_df
)

y_validation = validation_df["fraudulent"]

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
def cross_validate_text_pipeline(
    model,
    text,
    targets,
    threshold=0.60,
    number_of_folds=5
):
    splitter = StratifiedKFold(
        n_splits=number_of_folds,
        shuffle=True,
        random_state=42
    )

    results = []

    for fold, (train_indices, validation_indices) in enumerate(
        splitter.split(text, targets),
        start=1
    ):
        fold_model = clone(model)

        fold_model.fit(
            text.iloc[train_indices],
            targets.iloc[train_indices]
        )

        scores = fold_model.predict_proba(
            text.iloc[validation_indices]
        )[:, 1]

        predictions = (
            scores >= threshold
        ).astype(int)

        actual = targets.iloc[validation_indices]

        results.append({
            "fold": fold,
            "precision": precision_score(
                actual,
                predictions,
                zero_division=0
            ),
            "recall": recall_score(
                actual,
                predictions,
                zero_division=0
            ),
            "f1": f1_score(
                actual,
                predictions,
                zero_division=0
            )
        })

    return pd.DataFrame(results)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


no_stopword_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 1),
            stop_words=None
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

print("Pipeline created successfully")

In [ ]:
no_stopword_cv_table = cross_validate_text_pipeline(
    no_stopword_pipeline,
    train_text,
    y_train,
    threshold=0.60
).round(3)

no_stopword_cv_summary = (
    no_stopword_cv_table[
        ["precision", "recall", "f1"]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

no_stopword_cv_summary

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


custom_stop_words = list(
    ENGLISH_STOP_WORDS
    - {"no", "not", "nor", "never"}
)

negation_preserving_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 1),
            stop_words=custom_stop_words
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

In [ ]:
negation_cv_table = cross_validate_text_pipeline(
    negation_preserving_pipeline,
    train_text,
    y_train,
    threshold=0.60
).round(3)

negation_cv_summary = (
    negation_cv_table[
        ["precision", "recall", "f1"]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

negation_cv_summary

In [ ]:
negation_bigram_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words=custom_stop_words
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

In [ ]:
negation_bigram_cv_table = cross_validate_text_pipeline(
    negation_bigram_pipeline,
    train_text,
    y_train,
    threshold=0.60
).round(3)

negation_bigram_cv_summary = (
    negation_bigram_cv_table[
        ["precision", "recall", "f1"]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

negation_bigram_cv_summary

In [ ]:
comparison = pd.DataFrame({
    "negation_unigram": (
        negation_cv_table.mean(numeric_only=True)
    ),
    "negation_bigram": (
        negation_bigram_cv_table.mean(numeric_only=True)
    )
}).round(3)

comparison

In [ ]:
import numpy as np

from sklearn.base import clone
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold


def cross_validated_error_counts(
    pipeline,
    text,
    targets,
    threshold=0.60
):
    splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    out_of_fold_predictions = np.zeros(
        len(targets),
        dtype=int
    )

    for train_indices, test_indices in splitter.split(
        text,
        targets
    ):
        fold_pipeline = clone(pipeline)

        fold_pipeline.fit(
            text.iloc[train_indices],
            targets.iloc[train_indices]
        )

        fold_scores = fold_pipeline.predict_proba(
            text.iloc[test_indices]
        )[:, 1]

        out_of_fold_predictions[test_indices] = (
            fold_scores >= threshold
        ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        targets,
        out_of_fold_predictions
    ).ravel()

    return {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "error_cost": int(fp + 5 * fn)
    }

In [ ]:
error_comparison = pd.DataFrame([
    {
        "model": "negation_unigram",
        **cross_validated_error_counts(
            negation_preserving_pipeline,
            train_text,
            y_train
        )
    },
    {
        "model": "negation_bigram",
        **cross_validated_error_counts(
            negation_bigram_pipeline,
            train_text,
            y_train
        )
    }
])

error_comparison

In [ ]:
from sklearn.metrics import classification_report
negation_bigram_pipeline.fit(
    train_text,
    y_train
)

validation_scores = (
    negation_bigram_pipeline.predict_proba(
        validation_text
    )[:, 1]
)

validation_predictions = (
    validation_scores >= 0.60
).astype(int)

print(
    confusion_matrix(
        y_validation,
        validation_predictions
    )
)

print(
    classification_report(
        y_validation,
        validation_predictions,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)

In [ ]:
negation_preserving_pipeline.fit(
    train_text,
    y_train
)

negation_unigram_validation_scores = (
    negation_preserving_pipeline.predict_proba(
        validation_text
    )[:, 1]
)

negation_unigram_validation_predictions = (
    negation_unigram_validation_scores >= 0.60
).astype(int)

print(
    confusion_matrix(
        y_validation,
        negation_unigram_validation_predictions
    )
)

print(
    classification_report(
        y_validation,
        negation_unigram_validation_predictions,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)